<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day5/ExerciseXP/mini_projet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mini-projet : Assistant d’analyse des sentiments avec optimisation de BERT

In [ ]:
# 1. Mise à jour de protobuf pour aligner le moteur d'exécution (Runtime) avec le Gencode
%pip install --quiet --upgrade protobuf >=5.29.6

# 2. Recharger et forcer la liaison TensorFlow complète pour Transformers
%pip install --quiet --upgrade transformers[tf] datasets tensorflow-datasets


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-cloud-aiplatform 1.157.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which i

In [1]:
# =====================================================================
# Mini-projet BERT — version finale CPU rapide
# =====================================================================

import os, platform, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
import transformers

# --- Étape 0 : Configuration ---
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {DEVICE}")

MAX_LENGTH = 64
BATCH_SIZE = 8
EPOCHS     = 1
MAX_TRAIN  = 500
MAX_TEST   = 100
MODEL_DIR  = "bert_sentiment_model"
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Étape 1 : Chargement IMDB ---
import tensorflow.keras as keras
print("\nChargement IMDB...")
(x_train_enc, y_train_raw), (x_test_enc, y_test_raw) = \
    keras.datasets.imdb.load_data(num_words=20000)

word_index    = keras.datasets.imdb.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word.update({0: "<PAD>", 1: "<START>", 2: "<UNK>", 3: "<UNUSED>"})

def decode_review(encoded):
    return " ".join(index_to_word.get(i, "<UNK>") for i in encoded)

train_texts  = [decode_review(x) for x in x_train_enc[:MAX_TRAIN]]
train_labels = y_train_raw[:MAX_TRAIN].tolist()
test_texts   = [decode_review(x) for x in x_test_enc[:MAX_TEST]]
test_labels  = y_test_raw[:MAX_TEST].tolist()
print(f"Train : {len(train_texts)} | Test : {len(test_texts)}")

# --- Étape 2 : Tokenisation ---
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LENGTH):
        print(f"  Tokenisation de {len(texts)} exemples...")
        self.encodings = tokenizer(
            texts, max_length=max_len, padding="max_length",
            truncation=True, return_attention_mask=True,
            return_token_type_ids=True, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
        print(f"  ✅ Shape : {self.encodings['input_ids'].shape}")

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "token_type_ids": self.encodings["token_type_ids"][idx],
            "label":          self.labels[idx],
        }

train_dataset = IMDBDataset(train_texts, train_labels)
test_dataset  = IMDBDataset(test_texts,  test_labels)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Batches — train : {len(train_loader)} | test : {len(test_loader)}")

# --- Étape 3 : Modèle ---
print("\nChargement BERT...")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(DEVICE)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = max(1, total_steps // 10)
optimizer    = AdamW(model.parameters(), lr=2e-5, eps=1e-8, weight_decay=0.01)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"Steps : {total_steps} | Warmup : {warmup_steps}")

# --- Étape 4 : Entraînement ---
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for step, batch in enumerate(loader):
        ids, mask, ttype, labs = (
            batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE),
            batch["token_type_ids"].to(DEVICE), batch["label"].to(DEVICE)
        )
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype, labels=labs)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
        correct    += (out.logits.argmax(-1) == labs).sum().item()
        total      += labs.size(0)
        if (step + 1) % 10 == 0:
            print(f"  Step {step+1}/{len(loader)} | Loss: {total_loss/(step+1):.4f} | Acc: {correct/total:.4f}")
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            ids, mask, ttype, labs = (
                batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE),
                batch["token_type_ids"].to(DEVICE), batch["label"].to(DEVICE)
            )
            out = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype, labels=labs)
            total_loss += out.loss.item()
            correct    += (out.logits.argmax(-1) == labs).sum().item()
            total      += labs.size(0)
    return total_loss / len(loader), correct / total

best_val_acc    = 0.0
best_model_path = os.path.join(MODEL_DIR, "best_model.pt")

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}\n  ÉPOQUE {epoch}/{EPOCHS}\n{'='*50}")
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler)
    v_loss,  v_acc  = eval_epoch(model, test_loader)
    print(f"\n  Train → Loss: {tr_loss:.4f} | Acc: {tr_acc:.4f}")
    print(f"  Val   → Loss: {v_loss:.4f}   | Acc: {v_acc:.4f}")
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ Sauvegardé (val_acc={best_val_acc:.4f})")

model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

# --- Étape 5 : Évaluation finale ---
test_loss, test_acc = eval_epoch(model, test_loader)
print(f"\nTest Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

# --- Étape 6 : Assistant d'inférence ---
class SentimentAssistant:
    LABELS = {0: "Négatif ❌", 1: "Positif ✅"}

    def __init__(self, trained_model, bert_tokenizer, max_len=MAX_LENGTH):
        self.model = trained_model; self.tokenizer = bert_tokenizer
        self.max_len = max_len; self.model.eval()

    def predict(self, texts):
        if isinstance(texts, str): texts = [texts]
        encoded = self.tokenizer(
            texts, max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        with torch.no_grad():
            probs = torch.softmax(self.model(**encoded).logits, dim=-1).cpu().numpy()
        return [{"text": t, "label": self.LABELS[int(p.argmax())],
                 "confidence": round(float(p.max()), 4)} for t, p in zip(texts, probs)]

    def print_results(self, results):
        print("\n" + "="*65)
        for r in results:
            print(f"\n  Phrase     : {r['text']}")
            print(f"  Prédiction : {r['label']} ({r['confidence']:.1%})")
        print("="*65)

assistant = SentimentAssistant(model, tokenizer)
results   = assistant.predict([
    "The onboarding emails were confusing, but the agent fixed everything politely.",
    "Absolutely terrible experience. I will never use this service again.",
    "Outstanding product quality and lightning-fast delivery. Highly recommend!",
    "It was okay, nothing special but did the job.",
])
assistant.print_results(results)

Python       : 3.12.13
PyTorch      : 2.11.0+cu128
Transformers : 5.12.0
Device       : cuda

Chargement IMDB...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Train : 500 | Test : 100


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Steps : 63 | Warmup : 6

  ÉPOQUE 1/1
  Step 10/63 | Loss: 0.7088 | Acc: 0.4125
  Step 20/63 | Loss: 0.7094 | Acc: 0.4500
  Step 30/63 | Loss: 0.7029 | Acc: 0.4833
  Step 40/63 | Loss: 0.6986 | Acc: 0.4938
  Step 50/63 | Loss: 0.6938 | Acc: 0.5150
  Step 60/63 | Loss: 0.6926 | Acc: 0.5146

  Train → Loss: 0.6909 | Acc: 0.5240
  Val   → Loss: 0.6500   | Acc: 0.6600
  ✅ Sauvegardé (val_acc=0.6600)

Test Loss: 0.6500 | Test Accuracy: 0.6600


  Phrase     : The onboarding emails were confusing, but the agent fixed everything politely.
  Prédiction : Négatif ❌ (53.4%)

  Phrase     : Absolutely terrible experience. I will never use this service again.
  Prédiction : Négatif ❌ (54.9%)

  Phrase     : Outstanding product quality and lightning-fast delivery. Highly recommend!
  Prédiction : Positif ✅ (52.5%)

  Phrase     : It was okay, nothing special but did the job.
  Prédiction : Négatif ❌ (56.6%)
